# Data profiling - historic TfL dataset

### Inspection

In [7]:
# connect to the database
from pathlib import Path
import duckdb
import pandas as pd

# project directory
ROOT = Path.cwd()

data_folder = ROOT/"data"
if not data_folder.exists():
    ROOT = ROOT.parent

DB_PATH = ROOT /"data"/"processed"/"routeradar.duckdb"

# open database connection
conn = duckdb.connect(str(DB_PATH), read_only = True)

# inspect the raw table
conn.execute("""
SELECT *
FROM raw_service_status
LIMIT 5
""").df()



,Line,Start Date/Time,End Date/Time,Service Status,Status Message Details
0,Northern,01/04/2022 04:25:01,01/04/2022 04:27:46,Part Closure,No service between Moorgate and Kennington due...
1,Northern,01/04/2022 04:27:46,01/04/2022 11:01:45,Part Closure,No service between Moorgate and Kennington due...
2,Bakerloo,01/04/2022 05:19:49,01/04/2022 05:28:20,Part Suspended,No service between Queens Park and Harrow & We...
3,Bakerloo,01/04/2022 05:28:20,01/04/2022 05:41:53,Part Suspended,No service between Queens Park and Harrow & We...
4,Bakerloo,01/04/2022 05:41:53,01/04/2022 05:42:00,Part Suspended,No service between Queens Park and Harrow & We...


### Data Quality Assessment

In [9]:
quality = conn.execute("""
SELECT COUNT(*) AS total_records,

COUNT(*) FILTER(
WHERE "Status Message Details" IS NULL
OR TRIM("Status Message Details") = '' 
) AS missing_descriptions,

COUNT(*) FILTER(
WHERE TRY_STRPTIME("Start Date/Time", '%d/%m/%Y %H:%M:%S') IS NULL
) AS invalid_start_dates,

COUNT(*) FILTER(
WHERE TRY_STRPTIME("End Date/Time", '%d/%m/%Y %H:%M:%S') IS NULL
) AS invalid_end_dates,

FROM raw_service_status
""").df()

display(quality)

,total_records,missing_descriptions,invalid_start_dates,invalid_end_dates
0,118595,3467,0,0


### Duplicate records|

In [10]:
duplicates = conn.execute("""
SELECT "Line",
"Start Date/Time",
"End Date/Time",
"Service Status",
"Status Message Details",
COUNT(*) AS occurrences

FROM raw_service_status

GROUP BY ALL
HAVING COUNT(*)>1

ORDER BY occurrences DESC
LIMIT 20
""").df()

display(duplicates)

,Line,Start Date/Time,End Date/Time,Service Status,Status Message Details,occurrences
0,District,23/01/2023 17:54:19,23/01/2023 17:58:42,Part Suspended,"No service between Richmond, Ealing Broadway a...",12
1,District,25/01/2026 14:36:29,25/01/2026 14:49:14,Minor Delays,No service between Earl's Court and Kensington...,9
2,District,25/01/2026 14:49:15,25/01/2026 14:49:42,Minor Delays,No service between Earl's Court and Kensington...,9
3,District,25/01/2026 14:49:42,25/01/2026 15:36:08,Minor Delays,No service between Edgware Road and High Stree...,9
4,District,23/01/2023 17:53:57,23/01/2023 17:54:19,Part Suspended,"No service between Richmond, Ealing Broadway a...",8
5,District,25/01/2026 13:29:38,25/01/2026 13:45:46,Minor Delays,District line: No service between Edgware Road...,8
6,District,25/01/2026 13:45:46,25/01/2026 14:28:54,Minor Delays,No service between Edgware Road and High Stree...,8
7,Piccadilly,18/06/2026 20:16:34,18/06/2026 20:16:43,Severe Delays,No service between Hammersmith AND King's Cro...,6
8,Central,12/08/2026 11:38:38,12/08/2026 11:38:57,Minor Delays,Minor delays between North Acton and Ealing Br...,6
9,Piccadilly,19/04/2026 23:43:51,19/04/2026 23:45:24,Severe Delays,Severe delays between Acton Town and Heathrow ...,6


### Overlapping records
There are many occurences of exact repeating rows. The highest record has been repeated 12 times.
Here I make sure to remove exact duplicates to make sure I examine genuine interval relationships rather than comparing copies of the exact same row.

In [15]:
overlaps = conn.execute("""
WITH parsed AS(
SELECT DISTINCT
TRIM("Line") AS line,
TRY_STRPTIME("Start Date/Time", '%d/%m/%Y %H:%M:%S') AS start_time,
TRY_STRPTIME("End Date/Time", '%d/%m/%Y %H:%M:%S') AS end_time,
"Service Status" AS service_status,
"Status Message details" AS status_message

FROM raw_service_status
),

valid AS(
SELECT *
FROM parsed
WHERE start_time IS NOT NULL
AND end_time > start_time
),

ordered AS(
SELECT *,
MAX(end_time) OVER(
PARTITION BY line
ORDER BY start_time, end_time
ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
) AS previous_max_end
FROM valid
)

SELECT
line,
start_time,
end_time,
previous_max_end,
status_message

FROM ordered
WHERE start_time < previous_max_end

ORDER BY line, start_time
LIMIT 20
""").df()

display(overlaps)

,line,start_time,end_time,previous_max_end,status_message
0,Bakerloo,2022-04-04 10:54:08,2022-04-04 11:03:14,2022-04-04 11:03:14,No service between Queen's Park and Harrow & W...
1,Bakerloo,2022-04-04 11:03:14,2022-04-04 11:16:28,2022-04-04 11:16:28,No service between Queen's Park and Harrow & W...
2,Bakerloo,2022-04-10 16:32:54,2022-04-10 17:10:10,2022-04-10 17:10:09,No service between Queen's Park and Harrow & W...
3,Bakerloo,2022-04-23 11:13:19,2022-04-23 14:19:10,2022-04-23 14:19:08,Severe delays between Queen's Park and Harrow ...
4,Bakerloo,2022-04-23 20:04:12,2022-04-23 20:11:12,2022-04-23 20:11:12,No service between Queen's Park and Harrow & W...
5,Bakerloo,2022-04-23 20:15:18,2022-04-23 20:15:46,2022-04-23 20:15:46,Severe delays between Queen's Park and Harrow ...
6,Bakerloo,2022-04-23 20:15:46,2022-04-23 20:29:12,2022-04-23 20:29:12,Severe delays between Queen's Park and Harrow ...
7,Bakerloo,2022-05-03 21:09:46,2022-05-03 21:13:52,2022-05-03 21:13:52,No service between Queen's Park and Elephant &...
8,Bakerloo,2022-05-03 21:13:52,2022-05-03 21:20:34,2022-05-03 21:20:34,No service between Queen's Park and Elephant &...
9,Bakerloo,2022-05-03 21:20:34,2022-05-03 21:54:34,2022-05-03 21:54:34,No service between Paddington and Elephant & C...


The historical TfL dataset does not contain unique incident or train identifiers. This means that multiple records with similar descriptions and overlapping or consecutive timestamps may refer to the same underlying disruption. This cannot be confirmed from the data alone.

For RouteRadar, I will distinguish between exact duplicate records, potentially related status messages and overlapping disruption intervals.

Exact duplicate records will be removed from the analytical layer. Overlapping intervals will be merged for calculating the duration of the reported line disruption, without assuming that the associated messages represent the same disruption.

This approach avoids double counting the disruption duration while also preserving the distinction between reported status and individual disruption.